In [1]:
!pip install --upgrade transformers torch einops

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
import pandas as pd

# Model setup
model_name = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=dtype, trust_remote_code=True
).to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded!\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded!



In [3]:
def get_top_n_predictions(logits, n=5):
    probs = F.softmax(logits, dim=-1) # last dim which is Vocab.
    top_probs, top_ids = torch.topk(probs, n)
    return [(tokenizer.decode([int(i)]), float(p)) for i, p in zip(top_ids, top_probs)]

def visualize_step(step, text, preds, chosen):
    print("\n" + "=" * 80)
    print(f"STEP {step}")
    print(f"Current text: \"{text}\"")
    print("\nTop-n-next-token prediction:")

    df = pd.DataFrame({
        "Rank": range(1, 6),
        "Token": [repr(t) for t, _ in preds],
        "Probability": [f"{p:.4f}" for _, p in preds]
    })
    print(df.to_string(index=False))
    print(f"\nSelected token: {repr(chosen)}")

In [4]:
def generate_step_by_step(
        prompt,
        steps=5
):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    # Prompt -> [[ 785, 8251, 7578,  389,  279]] --> Shape of (1, 5)
    # i.e. 1 batch 5 tokens [batch size, seq_len]
    print(input_ids, input_ids.shape)
    current_text = prompt
    print("=" * 80)
    print("AUTOREGRESSIVE GENERATION (GREEDY)")
    print("=" * 80)
    print(f"Prompt: \"{prompt}\"")

    for step in range(steps):
        with torch.no_grad():# Runs Forward pass
            # model(input_ids) --> (batch_size, seq_len, vocab_size) : (1, 5, V)
            # batch -> 0 
            # -? Input (1, 5, 768) - 768 dim vector : After embedding layer
            # (batch=1, seq_len=5, hidden_dim=768)
            # Self Attention - ffn layer -> (1, 5, 768)
            # Final Layer = logits = W@X +b , Here W - (Vocab Size, 768), b - (Vocab Size)
            # Shaape of X - (768, )  This is after extract last pos [0, -1]
            # Hence final layer -> (Vocab Size, )
            logits = model(input_ids).logits[0, -1] # extract last token's logits
            # as last token's position has full context. Causality
            # information Odering 

        next_id = torch.argmax(logits).view(1, 1) # find index with max logit value
        # view reshapes to (1, 1) for concatenations, Reshapes [[next id]]
        display_logits = logits

        preds = get_top_n_predictions(display_logits)
        token_str = tokenizer.decode(next_id[0]) 
        # extracts the token id and convert to string

        visualize_step(step + 1, current_text, preds, token_str)

        input_ids = torch.cat([input_ids, next_id], dim=-1)
        current_text += token_str

    print("\nFINAL OUTPUT:")
    print(current_text)
    print("=" * 80 + "\n")

# Greedy
generate_step_by_step(
    "The cat sat on the",
    steps=5,
)

tensor([[ 785, 8251, 7578,  389,  279]], device='cuda:0') torch.Size([1, 5])
AUTOREGRESSIVE GENERATION (GREEDY)
Prompt: "The cat sat on the"

STEP 1
Current text: "The cat sat on the"

Top-n-next-token prediction:
 Rank     Token Probability
    1    ' mat'      0.8433
    2    ' cat'      0.0234
    3  ' fence'      0.0112
    4      '\n'      0.0094
    5 ' window'      0.0048

Selected token: ' mat'

STEP 2
Current text: "The cat sat on the mat"

Top-n-next-token prediction:
 Rank   Token Probability
    1     '.'      0.2194
    2   '.\n'      0.1605
    3     ','      0.1021
    4  ' and'      0.0859
    5 '.\n\n'      0.0659

Selected token: '.'

STEP 3
Current text: "The cat sat on the mat."

Top-n-next-token prediction:
 Rank   Token Probability
    1  ' The'      0.1801
    2     ' '      0.1774
    3 ' What'      0.0717
    4    ' ('      0.0448
    5   ' It'      0.0344

Selected token: ' The'

STEP 4
Current text: "The cat sat on the mat. The"

Top-n-next-token prediction:


In [5]:
def generate_beam_step_by_step(
    prompt,
    steps=5,
    num_beams=3,
    length_penalty=1.0
):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    # Each beam: (input_ids, cumulative_logprob)
    beams = [(input_ids, 0.0)]

    print("=" * 80)
    print("AUTOREGRESSIVE GENERATION (BEAM SEARCH)")
    print("=" * 80)
    print(f"Prompt: \"{prompt}\"")
    print(f"num_beams: {num_beams}, length_penalty: {length_penalty}\n")

    for step in range(steps):
        all_candidates = []

        print(f"\n{'='*30} STEP {step+1} {'='*30}")

        for beam_idx, (beam_ids, beam_score) in enumerate(beams):
            with torch.no_grad():
                logits = model(beam_ids).logits[0, -1]

            log_probs = F.log_softmax(logits, dim=-1)
            top_log_probs, top_ids = torch.topk(log_probs, num_beams)

            print(f"\nBeam {beam_idx+1} prefix:")
            print(tokenizer.decode(beam_ids[0], skip_special_tokens=True))

            for log_p, token_id in zip(top_log_probs, top_ids):
                new_ids = torch.cat(
                    [beam_ids, token_id.view(1, 1)], dim=-1
                )
                new_score = beam_score + log_p.item()

                all_candidates.append((new_ids, new_score))

                print(
                    f"  + {repr(tokenizer.decode([int(token_id)]))} "
                    f"(logP={log_p.item():.4f})"
                )

        # Rank all candidates
        all_candidates.sort(
            key=lambda x: x[1] / (x[0].shape[-1] ** length_penalty),
            reverse=True
        )

        # Keep top beams
        beams = all_candidates[:num_beams]

        print("\nSelected beams:")
        for i, (ids, score) in enumerate(beams):
            print(
                f"[{i+1}] score={score:.4f} | "
                f"{tokenizer.decode(ids[0], skip_special_tokens=True)}"
            )

    print("\nFINAL OUTPUT (BEST BEAM):")
    best_ids, best_score = beams[0]
    print(tokenizer.decode(best_ids[0], skip_special_tokens=True))
    print("=" * 80 + "\n")



# Beam
generate_beam_step_by_step(
    "The cat sat on the",
    steps=5,
    num_beams=3
)

AUTOREGRESSIVE GENERATION (BEAM SEARCH)
Prompt: "The cat sat on the"
num_beams: 3, length_penalty: 1.0


============================== STEP 1 ==============================

Beam 1 prefix:
The cat sat on the
  + ' mat' (logP=-0.1703)
  + ' cat' (logP=-3.7559)
  + ' fence' (logP=-4.4922)

Selected beams:
[1] score=-0.1703 | The cat sat on the mat
[2] score=-3.7559 | The cat sat on the cat
[3] score=-4.4922 | The cat sat on the fence

============================== STEP 2 ==============================

Beam 1 prefix:
The cat sat on the mat
  + '.' (logP=-1.5166)
  + '.\n' (logP=-1.8291)
  + ',' (logP=-2.2832)

Beam 2 prefix:
The cat sat on the cat
  + ' on' (logP=-1.5068)
  + ',' (logP=-2.8047)
  + ' hat' (logP=-3.2422)

Beam 3 prefix:
The cat sat on the fence
  + ',' (logP=-0.5049)
  + ' and' (logP=-2.6523)
  + '.' (logP=-3.1211)

Selected beams:
[1] score=-1.6869 | The cat sat on the mat.
[2] score=-1.9994 | The cat sat on the mat.

[3] score=-2.4535 | The cat sat on the mat,

======